# Part 2: Data Exploration & Preparation

This notebook explores the dataset used for predicting customer ratings (1–5 stars) using both structured data and review text.
The goal is to understand data patterns, quality issues, and relationships before feature engineering and modeling.

## 2.1 Structured Data Exploratory Data Analysis (EDA)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set pandas display option to show no more than 2 decimal places
pd.options.display.float_format = '{:.2f}'.format

# Define the URL for the training dataset
data_url = 'data/processed/train_clean.csv'

# Load the dataset into a pandas DataFrame
df = pd.read_csv(data_url)

# 1. Basic Information
print("--- Dataset Info ---")
display(df.info())

# 2. Check for Missing Values
print("\n--- Missing Values ---")
display(df.isnull().sum())

# 3. Descriptive Statistics for Numerical Features
print("\n--- Numerical Summary ---")
display(df.describe().round(2))

# 4. Descriptive Statistics for Categorical Features
print("\n--- Categorical Summary ---")
display(df.describe(include=['object', 'bool']))

# 5. Class Imbalance Analysis
print("\n--- Class Imbalance (Rating Distribution) ---")
rating_counts = df['rating'].value_counts().sort_index()
rating_pct = df['rating'].value_counts(normalize=True).sort_index() * 100
imbalance_df = pd.DataFrame({'Count': rating_counts, 'Percentage (%)': rating_pct})
display(imbalance_df)

# 6. Feature Impact Analysis - Visualizations
fig, axes = plt.subplots(3, 2, figsize=(16, 18))

# Plot 1: Delivery Days vs Rating
sns.boxplot(ax=axes[0, 0], data=df, x='rating', y='delivery_days', hue='rating', palette='viridis', legend=False)
axes[0, 0].set_title('Delivery Days by Rating')

# Plot 2: Product Price vs Rating
sns.boxplot(ax=axes[0, 1], data=df, x='rating', y='product_price', hue='rating', palette='magma', legend=False)
axes[0, 1].set_title('Product Price by Rating')

# Plot 3: Seller Rating vs Rating
sns.boxplot(ax=axes[1, 0], data=df, x='rating', y='seller_rating', hue='rating', palette='plasma', legend=False)
axes[1, 0].set_title('Seller Rating by Customer Rating')

# Plot 4: Return Initiated vs Rating
return_rating = pd.crosstab(df['rating'], df['return_initiated'], normalize='index') * 100
return_rating.plot(kind='bar', stacked=True, ax=axes[1, 1], color=['#66c2a5', '#fc8d62'])
axes[1, 1].set_title('Return Initiated % by Rating')
axes[1, 1].set_ylabel('Percentage')
axes[1, 1].legend(title='Return Initiated', labels=['No', 'Yes'])

# Plot 5: Discount Percent vs Rating
sns.boxplot(ax=axes[2, 0], data=df, x='rating', y='discount_pct', hue='rating', palette='rocket', legend=False)
axes[2, 0].set_title('Discount Percentage by Rating')

# Remove the empty 6th subplot
fig.delaxes(axes[2, 1])

plt.tight_layout()
plt.show()

# 7. Correlation Heatmap
plt.figure(figsize=(12, 8))
numeric_cols = df.select_dtypes(include=[np.number]).columns
correlation_matrix = df[numeric_cols].corr().round(2)
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Heatmap of Structured Features')
plt.show()

ModuleNotFoundError: No module named 'seaborn'

#### Observations

EDA was performed on the training dataset only to avoid data leakage. It still shows the patterns of the data, but avoid any knowledge of the test data to prevent any influence on the model.

This dataset contains 2000 rows and 14 features, with text as a feature. There are no missing values across all features.

Based on the rating distribution, it is clear that the dataset is imbalanced, and even though the majority is in the higher ranks, still 10.85% has rating of 1 and 14.60% has rating of 2, which will allow us to learn more about these types of customers to help the company achieve the goal.

The numerical summary shows all statistical information regarding each feature. Additionally, some features have graphs that show the relationship between a feature and rating. The correlation heatmap also helps show which features correlate with the rating.

For instnace, product_price shows that the price can be a minimum of 5 and a maximum is 500, which is a big range, and it may lead to the point that there are various products and they are not in the same categories. However, based on the graph, price does not affect the rating very much, since some high-priced items have different ratings.

Another example is delivery_days, which shows that delivery minimum is 1 day and the maximum is 25, but the average is about 4 days, and the majority of delivery days are between 1 and 5 days. In the graph, we can see that longer delivery days are mostly in ratings 2, 3, and some 4, which may indicate that this feature somewhat affects the rating. In the heatmap, the correlation is -0.26 between delivery days and rating which means that it is a weak negative relationship between them (as delivery days increas, rating is decreasing). Relationship is not strong, more on the weaker side.

seller_ratings shows a correlation between seller rating and rating overall (0.3 in the correlation heatmap), which means that a lower seller rating most likely will have more negative reviews. The means of the rating is about 4, which is good, since it means that the majority is high or good ratings.

return_initiated shows that lower rating has way more returns than high rating products, and it means that it is a signal of dissatisfaction of customers. Mean of this feature shows that 14% is only returns.

discount_pct has a wide range from 0% till 50% with 25.06% as the mean. The graph does not really show the difference between discount and rating, so we can say that this feature does not have a strong correlation with satisfaction or dissatisfaction.


## 2.2 Text Data EDA

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the dataset to ensure 'df' is defined
data_url = 'data/processed/train_clean.csv'
df = pd.read_csv(data_url)

# 1. Vocabulary Size Analysis (Training Data)
raw_text = ' '.join(df['review_text'])
raw_vocab = set(raw_text.split())
clean_text = raw_text.lower()
for char in '.,!?:;"()':
    clean_text = clean_text.replace(char, '')
processed_vocab = set(clean_text.split())

print(f'--- Training Data Vocabulary Analysis ---')
print(f'Vocabulary size (raw): {len(raw_vocab)}')
print(f'Vocabulary size (after basic cleaning): {len(processed_vocab)}')

# 2. Most Common Terms Overall
print('\n--- Top 15 Most Common Terms Overall ---')
# Clean and split the entire corpus to find common words (len > 3 to avoid noise)
all_words = clean_text.split()
long_words = [w for w in all_words if len(w) > 3]
top_overall = pd.Series(long_words).value_counts().head(15)
display(top_overall.to_frame(name='Frequency'))

# 3. Word Count Distribution
df['word_count'] = df['review_text'].str.split().str.len()
plt.figure(figsize=(10, 5))
sns.histplot(df['word_count'], bins=20, kde=True, color='skyblue')
plt.title('Distribution of Review Word Counts')
plt.show()

# 4. Word Count Distribution by Rating
print('\n--- Word Count Distribution by Rating ---')
counts = df['rating'].value_counts().sort_index()
pct = (df['rating'].value_counts(normalize=True).sort_index() * 100).round(2)
display(pd.DataFrame({'Review Count': counts, 'Percentage (%)': pct}))

# 5. Top Words by Rating Class
print('\n--- Top Words by Rating ---')
for r in sorted(df['rating'].unique()):
    cat_text = ' '.join(df[df['rating'] == r]['review_text']).lower()
    for char in '.,!?:;"()':
        cat_text = cat_text.replace(char, '')
    cat_words = [w for w in cat_text.split() if len(w) > 3]
    top_cat = pd.Series(cat_words).value_counts().head(10).index.tolist()
    print(f'Rating {r}: {", ".join(top_cat)}')

# 6. Sample documents
print('\n--- Sample Reviews per Rating Category ---')
for r in sorted(df['rating'].unique()):
    sample = df[df['rating'] == r]['review_text'].iloc[0]
    print(f'\n[Rating {r}] Sample:\n{sample}')

#### Observations
After preprocessing (removing commas, lowecasing, etc. ), the vocabulary shows that it is 324 words, when the raw vocabulary was 411 words.

Most frequent terms make sense since they are words that are common in the reviews, and based on the output, we can see that they appear in many ratings. For instance, "after", "product", "quality", etc. TF-IDF will reduce the weight of these words.

Based on the graph and the table, we can see that the majority of reviews have a rating of 3 or 4, and low rating like 1 has 10.85%, which means that the data is imbalanced and needs to have recall, precision, and F-beta during the model evaluation. Also, 10.85% will allow us to see the patterns of dissatisfied customers and help to prevent it.

Top words by rating are now not very clear since top words are too common and identifcal. For example, "this", "quality", "product", "expectations" are repeating in many ratings, which indicate that there are a limited separation of words between ratings and it will be a challenge for TF-IDF.  
